## Wetland bird migration analysis

In [2]:
import numpy as np
import pandas as pd

np.random.seed(91)

wetlands = [
    "East Kolkata",
    "Nalban",
    "Santragachi",
    "Purbasthali",
    "Bethuadahari"
]

species = [
    "Bar-headed Goose",
    "Northern Shoveler",
    "Common Teal",
    "Little Egret",
    "Black-winged Stilt",
    "Pied Kingfisher",
    "Purple Swamphen"
]

dates = pd.date_range("2025-10-01", "2026-01-31", freq="D")

rows = []

for date in dates:
    for wetland in wetlands:
        # Not every species is observed at every wetland/date
        n_species = np.random.randint(3, 7)
        sampled_species = np.random.choice(
            species,
            size=n_species,
            replace=False
        )

        for sp in sampled_species:
            rows.append({
                "date": date,
                "wetland": wetland,
                "species": sp,
                "individuals": np.random.poisson(
                    np.random.uniform(8, 50)
                ),
                "observation_hours": np.random.uniform(1, 5)
            })

birds = pd.DataFrame(rows)

# Introduce missing observations
idx = np.random.choice(
    birds.index,
    size=30,
    replace=False
)

birds.loc[idx, "individuals"] = np.nan

birds.head()

,date,wetland,species,individuals,observation_hours
0,2025-10-01,East Kolkata,Black-winged Stilt,46.0,2.283253
1,2025-10-01,East Kolkata,Little Egret,35.0,1.786430
2,2025-10-01,East Kolkata,Bar-headed Goose,19.0,2.847342
3,2025-10-01,East Kolkata,Northern Shoveler,30.0,3.420595
4,2025-10-01,East Kolkata,Pied Kingfisher,9.0,4.672775


In [3]:
birds.groupby(['wetland','species'])['individuals'].mean()
birds.query('wetland == "Bethuadahari" & species == "Little Egret"')
birds['individuals'].where((birds['wetland']=='Bethuadahari') & (birds['species']=='Little Egret')).dropna().mean()

np.float64(26.373333333333335)

In [4]:
birds.groupby(['wetland','species'])['individuals'].mean()

wetland       species           
Bethuadahari  Bar-headed Goose      27.444444
              Black-winged Stilt    29.921053
              Common Teal           29.055556
              Little Egret          26.373333
              Northern Shoveler     32.363636
              Pied Kingfisher       29.743590
              Purple Swamphen       28.192771
East Kolkata  Bar-headed Goose      30.187500
              Black-winged Stilt    29.697674
              Common Teal           30.416667
              Little Egret          30.470588
              Northern Shoveler     29.678571
              Pied Kingfisher       27.078947
              Purple Swamphen       29.012048
Nalban        Bar-headed Goose      28.657895
              Black-winged Stilt    29.157143
              Common Teal           30.512821
              Little Egret          27.207317
              Northern Shoveler     26.813953
              Pied Kingfisher       26.485714
              Purple Swamphen       29.635294
P

1. Compare migration periods

Create a month column from date.

For each month, calculate:

- total individuals observed
- number of unique species
- mean observation effort
- mean individuals observed per hour

In [5]:
birds['month'] = birds['date'].dt.strftime('%B')
birds.groupby('month').apply(lambda df: pd.Series({
    'total_indv': df['individuals'].sum(),
    'no_of_uni_sp': df['species'].nunique(),
    'mean_obs_effort': df['observation_hours'].mean(),
    'mean_indv_obs_per_hr': (df['individuals']/df['observation_hours']).mean()
})).sort_values(by='mean_obs_effort',ascending=False)

,total_indv,no_of_uni_sp,mean_obs_effort,mean_indv_obs_per_hr
month,,,,
October,20192.0,7.0,3.015256,11.675302
November,19665.0,7.0,3.005085,11.553471
December,19302.0,7.0,2.979248,11.631758
January,20127.0,7.0,2.968054,11.951056


2. Compare wetlands

For each wetland, calculate:

- total individuals observed
- number of unique species
- mean individuals/hour
- 90th percentile of individuals/hour

In [6]:
birds.groupby('wetland').apply(lambda df: pd.Series({
    'total_indv': df['individuals'].sum(),
    'no_of_uni_sp': df['species'].nunique(),
    'mean_indv_per_hr': (df['individuals']/df['observation_hours']).mean(),
    'percentile_90': (df['individuals']/df['observation_hours']).quantile(0.9)
})).sort_values(by='mean_indv_per_hr',ascending=False)

,total_indv,no_of_uni_sp,mean_indv_per_hr,percentile_90
wetland,,,,
East Kolkata,16190.0,7.0,12.365137,24.900364
Purbasthali,15476.0,7.0,11.786240,23.483962
Santragachi,16036.0,7.0,11.543257,21.943945
Bethuadahari,16075.0,7.0,11.472431,22.031093
Nalban,15509.0,7.0,11.352546,20.765328


3. Detect unusually strong migration events

Create an effort-adjusted abundance column:

individuals_per_hour = individuals / observation_hours

Calculate the overall mean and standard deviation of this metric.

Define a sighting as an "unusually high event" when:

individuals_per_hour > mean + 2 × standard deviation

In [7]:
birds['indv_per_hr'] = birds['individuals']/birds['observation_hours']
birds['event_status']= np.where(
    (birds['indv_per_hr'] > (birds['indv_per_hr'].mean() + (2* birds['indv_per_hr'].std()))),
    'unusually high event',
    'normal'
)

In [53]:
df1 = pd.pivot_table(birds,index='wetland',columns='month',values='event_status',aggfunc='count').sort_values(by='October',ascending=False)
df1.reset_index().columns

Index(['wetland', 'December', 'January', 'November', 'October'], dtype='str', name='month')

In [54]:
pd.melt(
    df1.reset_index(),
    id_vars=['wetland'],
    value_vars=['December','January','November','October']
).sort_values(by=['wetland','month'])

,wetland,month,value
3,Bethuadahari,December,135
8,Bethuadahari,January,138
13,Bethuadahari,November,145
18,Bethuadahari,October,138
4,East Kolkata,December,141
9,East Kolkata,January,140
14,East Kolkata,November,138
19,East Kolkata,October,136
1,Nalban,December,136
6,Nalban,January,140


In [31]:
birds.groupby(['wetland','month'])['event_status'].count()

wetland       month   
Bethuadahari  December    135
              January     138
              November    145
              October     138
East Kolkata  December    141
              January     140
              November    138
              October     136
Nalban        December    136
              January     140
              November    137
              October     143
Purbasthali   December    137
              January     137
              November    136
              October     141
Santragachi   December    129
              January     140
              November    137
              October     148
Name: event_status, dtype: int64

conclusion: 

In [57]:
birds.head()
birds['new_month'] = birds['date'].dt.to_period('M')
birds.head()

,date,wetland,species,individuals,observation_hours,month,indv_per_hr,event_status,new_month
0,2025-10-01,East Kolkata,Black-winged Stilt,46.0,2.283253,October,20.146693,normal,2025-10
1,2025-10-01,East Kolkata,Little Egret,35.0,1.786430,October,19.592149,normal,2025-10
2,2025-10-01,East Kolkata,Bar-headed Goose,19.0,2.847342,October,6.672890,normal,2025-10
3,2025-10-01,East Kolkata,Northern Shoveler,30.0,3.420595,October,8.770405,normal,2025-10
4,2025-10-01,East Kolkata,Pied Kingfisher,9.0,4.672775,October,1.926050,normal,2025-10
